# Module 17: Geospatial Ride Sharing Dispatch Uber — Interactive Laboratory

Every cell below runs the module's **real** implementation from
`project_solution/geospatial_dispatch.py`. Nothing here prints a claim it has not verified.

What you will do:

1. Load the engine and inspect what it actually exports.
2. Run its primary workflow and check the assertions that define correctness.
3. **Commit to a prediction**, then run the cell that tests it.
4. Measure a property rather than asserting one.
5. Fix a deliberately broken cell in place.

> The code in cells 4, 6 and 8 is lifted from this module's own test suite, so it
> cannot drift from the implementation. If the API changes, those tests fail
> first and this notebook is regenerated from them.


## 1. Load the engine and introspect it

Rather than trusting a hardcoded list of class names, ask the module what it
actually contains.


In [ ]:
import inspect
import sys
from pathlib import Path

sys.path.insert(0, str(Path('.').resolve() / 'project_solution'))
import geospatial_dispatch

classes = [n for n, o in inspect.getmembers(geospatial_dispatch, inspect.isclass)
           if o.__module__ == 'geospatial_dispatch']
functions = [n for n, o in inspect.getmembers(geospatial_dispatch, inspect.isfunction)
             if o.__module__ == 'geospatial_dispatch']

print('module   : geospatial_dispatch')
print(f'classes  : {classes}')
print(f'functions: {functions}')
print()
for name in classes:
    obj = getattr(geospatial_dispatch, name)
    try:
        sig = inspect.signature(obj.__init__)
        params = [p for p in sig.parameters if p != 'self']
    except (TypeError, ValueError):
        params = ['<builtin>']
    print(f'  {name}({", ".join(params)})')

## 2. Baseline: Geohash encoding consistency

This is the module's own `test_geohash_encoding_consistency` — real instantiation, real calls, real
assertions. If it runs clean, the property it encodes holds.


In [ ]:
from geospatial_dispatch import (
    Geohash,
    GeospatialIndex,
    haversine_distance_km,
)

lat, lon = 37.7749, -122.4194
gh = Geohash.encode(lat, lon, precision=6)
assert len(gh) == 6
# San Francisco Geohash begins with '9q8yy'
assert gh.startswith("9q8y")

print('PASSED: test_geohash_encoding_consistency')

## 3. 🔮 Prediction — commit before you run

Predict how many geohash cells a 5 km radius search must query, and why a single cell lookup misses drivers that are 100 m away.

Write your answer down. An uncommitted guess teaches nothing, because you will
retro-fit it to whatever the next cell prints.

The next cell runs `test_haversine_distance_calculation`, which tests exactly this property.


In [ ]:
dist = haversine_distance_km(40.7128, -74.0060, 39.9526, -75.1652)
# True distance is ~130 km
assert 120.0 < dist < 140.0

print('PASSED: test_haversine_distance_calculation')

## 4. Measure it: Driver location update and cell migration

An assertion tells you a property holds. A measurement tells you *how much*.
This cell runs `test_driver_location_update_and_cell_migration` and times it.


In [ ]:
import time

_t0 = time.perf_counter()

idx = GeospatialIndex(precision=5)
idx.update_driver_location("driver-1", 37.7749, -122.4194)

initial_cell = idx.driver_cell["driver-1"]
assert "driver-1" in idx.cells[initial_cell]

# Move driver to Oakland (across bay)
idx.update_driver_location("driver-1", 37.8044, -122.2712)
new_cell = idx.driver_cell["driver-1"]

assert new_cell != initial_cell
# Invariant: Driver must be removed from old cell and added to new cell
assert "driver-1" not in idx.cells[initial_cell]
assert "driver-1" in idx.cells[new_cell]

_elapsed = (time.perf_counter() - _t0) * 1000
print('PASSED: test_driver_location_update_and_cell_migration')
print(f'wall clock: {_elapsed:.2f} ms')

## 5. 🛠️ Fix this cell — it is deliberately broken

The cell below asserts something **false** about the real object. Read the
failure, work out the true value from the module's actual behaviour, and correct
the expected number.

Do not delete the assertion. The point is to make it pass by knowing the answer.


In [ ]:
# DELIBERATELY BROKEN - fix the expected value below.
# Hint: print the real value first, then decide what the assertion should say.

exports = [n for n in dir(geospatial_dispatch) if not n.startswith('_')]
print(f'actual export count: {len(exports)}')
print(f'actual exports     : {exports}')

EXPECTED_EXPORT_COUNT = 999      # <-- wrong on purpose. Replace it.

assert len(exports) == EXPECTED_EXPORT_COUNT, (
    f'expected {EXPECTED_EXPORT_COUNT} exports, found {len(exports)}. '
    'Read the printed value above and correct the constant.'
)
print('Fixed - assertion now reflects reality.')

### 🎓 Key takeaways

1. Geospatial indexing turns a 2-D range query into a 1-D prefix scan.
2. Cell boundaries mean neighbour cells must be queried too - always.
3. Dispatch is a matching problem under a moving supply distribution.

---

**Continue with this module:**

- [README.md](README.md) — the mental model and failure modes
- [PROJECT_GUIDE.md](PROJECT_GUIDE.md) — build it yourself, in 3 tiers
- [starter/](starter/) — your stubs; run the tests from there to grade yourself
- [debug_lab/SYMPTOMS.md](debug_lab/SYMPTOMS.md) — diagnose planted bugs from the symptom
- [TROUBLESHOOTING_AND_EDGE_CASES.md](TROUBLESHOOTING_AND_EDGE_CASES.md) — real errors, real causes
- [SELF_ASSESSMENT_AND_CHALLENGES.md](SELF_ASSESSMENT_AND_CHALLENGES.md) — quiz and diagnostics
